# 01 — Processed data and QC

Inspect `data/processed/perturbseq.h5ad` (Norman et al. 2019 Perturb-seq,
QC-filtered, 500 HVGs, library-size normalised) and the persisted
perturbation-level train/val/test split.

Run `scripts/download_data.py` -> `scripts/preprocess_data.py` first if the
file is missing.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("project root:", ROOT)


In [ ]:
import anndata as ad
import pandas as pd

adata = ad.read_h5ad(ROOT / "data/processed/perturbseq.h5ad")
adata

## Dataset overview

In [ ]:
print(f"cells: {adata.n_obs:,}")
print(f"genes (HVGs): {adata.n_vars:,}")
print(f"perturbations: {adata.obs['perturbation'].nunique()}")
print(f"layers: {list(adata.layers)}")
print(f"preprocessing: {adata.uns.get('preprocessing')}")

## QC distributions

The per-cell QC columns survive preprocessing so the filters can be audited.

In [ ]:
import matplotlib.pyplot as plt

qc_cols = [c for c in ("ncounts", "ngenes", "percent_mito") if c in adata.obs]
fig, axes = plt.subplots(1, len(qc_cols), figsize=(4 * len(qc_cols), 3.2))
for ax, col in zip(axes, qc_cols):
    ax.hist(adata.obs[col].to_numpy(), bins=60)
    ax.set_title(col)
    ax.set_xlabel(col)
fig.tight_layout()
out = ROOT / "results" / "figures" / "qc_distributions.png"
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=150)
print("saved", out)

## Perturbation-level split

Every cell of a given perturbation lands in exactly one split, so no
perturbation identity leaks across train/val/test.

In [ ]:
import json

split = json.loads((ROOT / "data/processed/splits/perturbation_split.json").read_text())
rows = [
    {"split": name,
     "cells": len(split[name]),
     "perturbations": len(split[f"{name}_perts"])}
    for name in ("train", "val", "test")
]
pd.DataFrame(rows)

In [ ]:
counts = adata.obs["perturbation"].value_counts()
print("top 15 perturbations by cell count")
print(counts.head(15).to_string())
print()
print(f"control cells: {counts.get('control', 0):,}")